In [ ]:
%pip install langchain-openai duckduckgo-search

In [2]:
import os
from dotenv import load_dotenv
from ddgs import DDGS
from langchain_core.tools import tool
from langchain_community.vectorstores import SKLearnVectorStore
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

load_dotenv()

C:\Users\HP\AppData\Local\Temp\ipykernel_18000\72364071.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import SKLearnVectorStore


True

In [3]:
load_dotenv()

# My_Personal_key
api_key= os.getenv("OPENAI_API_KEY")
base_url="https://api.groq.com/openai/v1"
chat_model_name = "openai/gpt-oss-20b"

# --- Embedding model (different server, different key) ---
embedding_api_key    = os.getenv("EMBEDDING_API_KEY")
embedding_base_url   = "https://qwen-embed.publicaai.com/v1"
embedding_model_name = "Qwen/Qwen3-Embedding-0.6B"

print("Chat base URL     :", base_url)
print("Chat model        :", chat_model_name)
print("Embed base URL    :", embedding_base_url)
print("Embedding model   :", embedding_model_name)
#print(api_key)

Chat base URL     : https://api.groq.com/openai/v1
Chat model        : openai/gpt-oss-20b
Embed base URL    : https://qwen-embed.publicaai.com/v1
Embedding model   : Qwen/Qwen3-Embedding-0.6B


In [33]:
llm = ChatOpenAI(
    api_key=api_key,
    base_url=base_url,
    model=chat_model_name,
    temperature=0.2  # Changed from 0 to 0.2 for natural web search synthesis
)

In [34]:
import os
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import SKLearnVectorStore

docs = []

# Load TXT data
if os.path.exists("../data/NFCMS_DATA.txt"):
    txt_loader = TextLoader("../data/NFCMS_DATA.txt", encoding="utf-8")
    docs.extend(txt_loader.load())

# Load PDF data
if os.path.exists("../data/NPHCDA.pdf"):
    pdf_loader = PyPDFLoader("../data/NPHCDA.pdf")
    docs.extend(pdf_loader.load())

print(f"Loaded {len(docs)} source documents.")

# Chunk texts
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
splits = text_splitter.split_documents(docs)
print(f"Created {len(splits)} document chunks.")

# Build and persist SKLearnVectorStore to data/health_index.json
store_path = "../data/health_index.json"

healthdb = SKLearnVectorStore.from_documents(
    documents=splits,
    embedding=embeddings,
    persist_path=store_path,
    serializer="json"
)

healthdb.persist()
print(f"Vector store successfully generated and saved to: {store_path}")

# Create retriever
health_retriever = healthdb.as_retriever(
    search_type="mmr", 
    search_kwargs={'k': 3, 'fetch_k': 10}
)

Loaded 2 source documents.
Created 53 document chunks.
Vector store successfully generated and saved to: ../data/health_index.json


In [35]:
from ddgs import DDGS
from langchain_core.tools import tool

@tool
def search_health_docs(query: str) -> str:
    """Searches the local CNITS health database for child nutrition, growth indicators, and immunization guidelines."""
    docs = health_retriever.invoke(query)
    if not docs:
        return "No relevant health documents found in the database."
    return "\n\n".join([d.page_content for d in docs])

@tool
def search_web(query: str) -> str:
    """Searches DuckDuckGo for live health updates, local clinic info, or general web inquiries."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(query, max_results=3))
            if not results:
                return "No web results found."
            return "\n\n".join([f"Title: {r['title']}\nSnippet: {r['body']}" for r in results])
    except Exception as e:
        return f"Web search failed: {str(e)}"

tools = [search_health_docs, search_web]
print(f"Loaded {len(tools)} tools.")

Loaded 2 tools.


In [36]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import InMemorySaver

memory = InMemorySaver()

system_prompt = (
    "You are an expert AI assistant for the Child Nutrition and Immunization Tracking System (CNITS) in Nigeria. "
    "Always search local health documents first using `search_health_docs` for immunization schedules, "
    "WHO growth standards, and nutrition guidelines. "
    "If information is missing or requires up-to-date real-time context, use `search_web`. "
    "Provide clear, accurate, and empathetic medical guidance."
)

agent_executor = create_react_agent(
    model=llm,
    tools=tools,
    prompt=system_prompt,
    checkpointer=memory
)

# Run test query
config = {"configurable": {"thread_id": "cnits_test_run"}}
user_query = "What is the recommended BCG vaccination timing for newborns in Nigeria, and what are its core indications?"

response = agent_executor.invoke(
    {"messages": [("user", user_query)]},
    config=config
)

print("\nCNITS Assistant Response:\n")
print(response["messages"][-1].content)


CNITS Assistant Response:

**BCG (Bacillus Calmette‑Guérin) vaccination for newborns in Nigeria**

| Item | Details |
|------|---------|
| **When it should be given** | • **At birth** (ideally within the first 24 hours) or < 7 days of life. <br>• If the baby is not vaccinated at birth, it can still be given up to 12 months of age, but the earlier the better for protection against severe forms of TB. |
| **How it is given** | • A single intradermal injection (usually 0.05 mL) in the upper arm or thigh. |
| **Core indications (why we give it)** | 1. **Prevention of severe, life‑threatening tuberculosis** in infants: <br>   • TB meningitis <br>   • TB miliary disease <br>   • Disseminated TB <br>2. **Universal protection** – BCG is recommended for all infants in countries where TB is common, regardless of birth weight or HIV status (unless contraindicated). |
| **Contraindications / cautions** | • **Severe immunodeficiency** (e.g., known HIV‑positive infant with low CD4 count). <br>• **A

In [25]:
config = {"configurable": {"thread_id": "cnits_test_run"}}

user_query = "What is the recommended BCG vaccination timing for newborns in Nigeria, and what are its core indications?"

print(f"User Query: {user_query}\n" + "-" * 50)

response = agent_executor.invoke(
    {"messages": [("user", user_query)]},
    config=config
)

print("\nCNITS Assistant Response:\n")
print(response["messages"][-1].content)

User Query: What is the recommended BCG vaccination timing for newborns in Nigeria, and what are its core indications?
--------------------------------------------------

CNITS Assistant Response:

**BCG (Bacille‑Calmette‑Guérin) vaccination in Nigeria**

| **What** | **Answer** |
|----------|------------|
| **Recommended timing** | **At birth** – ideally within the first 24 hours, but no later than 7 days after delivery. |
| **Core indications** | 1. **Prevention of severe childhood tuberculosis** (miliary TB, TB meningitis, TB of bones/joints). <br>2. **Universal protection in a high‑burden setting** – Nigeria’s TB incidence is high, so every newborn receives BCG to gain early immunity. <br>3. **Safe co‑administration** – BCG can be given with HepB, OPV, and Pentavalent at the same visit. <br>4. **HIV‑exposed infants** – WHO recommends BCG for all infants, including those born to HIV‑positive mothers, unless a severe immunodeficiency is known. |

---

### Practical points for health 

In [26]:
for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the recommended BCG vaccination timing for newborns in Nigeria, and what are its core indications?
================================== Ai Message ==================================
Tool Calls:
  search_health_docs (fc_e83ce086-1a80-46ef-881c-cf6654b8c65c)
 Call ID: fc_e83ce086-1a80-46ef-881c-cf6654b8c65c
  Args:
    query: BCG vaccination timing newborn Nigeria recommended schedule core indications
================================= Tool Message =================================
Name: search_health_docs

Keywords: breastfeeding, nursing mother, lactating women, milk supply, breast milk, nursing diet

From NFCMS 2021 national data:
- Mean energy intake for lactating women: 2,061 kcal/day (below recommended 2,500-2,800 kcal)
- Mean protein intake for lactating women: 53 g/day (below recommended 71 g)
- 66% of lactating women have inadequate protein intake (63% rural, 51% urban)
- Mean animal-source pr

In [37]:
# Turn 2: Follow-up question referencing the previous answer
followup_query = "What about the OPV 0 dose? When should that be given relative to BCG?"

response_2 = agent_executor.invoke(
    {"messages": [("user", followup_query)]},
    config=config
)

print("\nFollow-Up Response:\n")
print(response_2["messages"][-1].content)


Follow-Up Response:

**OPV 0 (Oral Polio Vaccine – first dose) in Nigeria**

| Item | Details |
|------|---------|
| **When it should be given** | • **At birth** – the very first dose of OPV is scheduled for the newborn’s first day of life (or within the first 24 hours). |
| **How it is given** | • **Oral** – a small drop of liquid vaccine is placed in the baby’s mouth (usually in the cheek or under the tongue). |
| **Relationship to BCG** | • **BCG and OPV 0 are usually given on the same visit** – most health facilities administer both vaccines together at the birth‑care appointment. <br>• The BCG injection is given intradermally (upper arm or thigh), while OPV 0 is given orally, so they can be given simultaneously without interference. |
| **Core indications** | 1. **Prevention of poliomyelitis** – the first dose protects the infant from paralytic polio and helps interrupt transmission. <br>2. **Universal coverage** – all newborns in Nigeria receive OPV 0 regardless of birth weight 

In [28]:
web_query = "What are the latest 2026 NPHCDA immunization guidelines or outbreak updates in Nigeria?"

response_3 = agent_executor.invoke(
    {"messages": [("user", web_query)]},
    config=config
)

print("\nWeb Fallback Response:\n")
print(response_3["messages"][-1].content)

C:\Users\HP\AppData\Local\Temp\ipykernel_18840\1612749005.py:16: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:
C:\Users\HP\AppData\Local\Temp\ipykernel_18840\1612749005.py:16: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Web Fallback Response:

**2026 NPHCDA Immunization Guidelines & Outbreak Updates (Nigeria)**  
*(Information compiled from the NPHCDA website, WHO‑Nigeria, and recent press releases.  For the most current, always check the official NPHCDA portal or your local health authority.)*

---

## 1.  Routine Immunization Schedule (2026)

| Vaccine | Dose | Age (weeks) | Notes |
|---------|------|-------------|-------|
| **BCG** | 0 | Birth | Given at the same visit as OPV 0 and HepB 0. |
| **OPV 0** | 0 | Birth | First oral polio dose. |
| **HepB 0** | 0 | Birth | First hepatitis‑B dose. |
| **Pentavalent (DTP‑HepB‑Hib)** | 1 | 6 weeks | First dose. |
| **OPV 1** | 1 | 6 weeks | |
| **Pentavalent 2** | 2 | 10 weeks | |
| **OPV 2** | 2 | 10 weeks | |
| **Pentavalent 3** | 3 | 14 weeks | |
| **OPV 3** | 3 | 14 weeks | |
| **IPV** | 3 | 14 weeks | Added to the routine schedule in 2025. |
| **Measles‑Rubella (MR)** | 1 | 9 months | New introduction in 2025; now part of the routine schedule. |
| **